# step3_feature_mlp

**Owner:** P4  
**Purpose:** Feature-based MLP (history, category, entities, article age), with feature ablation.  
**Inputs:** `data/processed/` from steps 1 and 3  
**Outputs:** `outputs/results_step3_feature_mlp.csv`

> Runtime: Colab. Run all cells top to bottom; keep outputs saved.

In [4]:
# ==========================================================
# 1. GOOGLE COLAB SETUP
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import subprocess
from pathlib import Path

# Clone the GitHub project into Colab
PROJECT_ROOT = Path("/content/BT4222-Project-Group-4--MIND")

if not (PROJECT_ROOT / "config.py").exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git",
        str(PROJECT_ROOT)
    ], check=True)

sys.path.insert(0, str(PROJECT_ROOT))

# Shared Google Drive project folder
DATA_ROOT = Path("/content/drive/MyDrive/BT4222 G4/BT4222_Project")

os.environ["MIND_PROJECT_DIR"] = str(DATA_ROOT)

import config
import importlib
importlib.reload(config)

PROCESSED_DIR = Path(config.PROCESSED_DIR)

print("Repository found:", (PROJECT_ROOT / "config.py").exists())
print("Project folder found:", DATA_ROOT.is_dir())
print("Processed folder found:", PROCESSED_DIR.is_dir())

os.environ["MIND_PROJECT_DIR"] = str(DATA_ROOT)

import config
import importlib
importlib.reload(config)

PROCESSED_DIR = Path(config.PROCESSED_DIR)

print("Repository found:", (PROJECT_ROOT / "config.py").exists())
print("Processed folder found:", PROCESSED_DIR.exists())


Mounted at /content/drive
Repository found: True
Project folder found: True
Processed folder found: True
Repository found: True
Processed folder found: True


In [5]:
# ==========================================================
# 2. NEW FEATURE MLP ARCHITECTURE
# ==========================================================

import torch
import torch.nn as nn

class FeatureMLP(nn.Module):
    def __init__(self, n_categories, n_subcategories):
        super().__init__()

        # Shared embeddings for users' history and candidate news
        self.category_embedding = nn.Embedding(
            n_categories, 8, padding_idx=0
        )
        self.subcategory_embedding = nn.Embedding(
            n_subcategories, 16, padding_idx=0
        )

        # 5 numerical + 24 user + 24 article = 53 inputs
        self.mlp = nn.Sequential(
            nn.Linear(53, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def average_history(self, history_ids, embedding):
        # Ignore padding and unknown category IDs (0)
        mask = (history_ids != 0).unsqueeze(-1)
        vectors = embedding(history_ids) * mask

        counts = mask.sum(dim=1).clamp(min=1)
        return vectors.sum(dim=1) / counts

    def forward(
        self,
        category_idx,
        subcategory_idx,
        history_category_idx,
        history_subcategory_idx,
        numerical_features
    ):
        # Article embeddings: 8 + 16 dimensions
        article_cat = self.category_embedding(category_idx)
        article_subcat = self.subcategory_embedding(
            subcategory_idx
        )

        # User history embeddings: 8 + 16 dimensions
        user_cat = self.average_history(
            history_category_idx,
            self.category_embedding
        )
        user_subcat = self.average_history(
            history_subcategory_idx,
            self.subcategory_embedding
        )

        # Combine all 53 dimensions
        x = torch.cat([
            numerical_features,
            user_cat,
            user_subcat,
            article_cat,
            article_subcat
        ], dim=1)

        return self.mlp(x).squeeze(-1)


print("Feature MLP architecture defined successfully!")

Feature MLP architecture defined successfully!


In [6]:

# ==========================================================
# 3. TEST FEATURE MLP WITH DUMMY DATA
# ==========================================================

import torch

torch.manual_seed(42)

# Example vocabulary sizes (only for testing)
model = FeatureMLP(
    n_categories=20,
    n_subcategories=300
)

model.eval()

# Test 5 candidate articles
category_idx = torch.tensor([1, 2, 3, 1, 2])
subcategory_idx = torch.tensor([10, 20, 30, 10, 20])

# Previously clicked article categories
# Zeros represent padding / unknown categories
history_category_idx = torch.tensor([
    [1, 2, 3, 0],
    [2, 2, 0, 0],
    [3, 1, 2, 1],
    [0, 0, 0, 0],
    [1, 1, 2, 0]
])

history_subcategory_idx = torch.tensor([
    [10, 20, 30, 0],
    [20, 20, 0, 0],
    [30, 10, 20, 10],
    [0, 0, 0, 0],
    [10, 10, 20, 0]
])

# Five numerical features per article
numerical_features = torch.rand(5, 5)

with torch.no_grad():
    scores = model(
        category_idx,
        subcategory_idx,
        history_category_idx,
        history_subcategory_idx,
        numerical_features
    )

print("Output shape:", scores.shape)
print("Scores:", scores)
print("All scores finite:", torch.isfinite(scores).all().item())

assert scores.shape == (5,)
assert torch.isfinite(scores).all()

print("\nMLP architecture test PASSED!")


Output shape: torch.Size([5])
Scores: tensor([0.2266, 0.2672, 0.2051, 0.2098, 0.2728])
All scores finite: True

MLP architecture test PASSED!


In [7]:

# ==========================================================
# 4. FIVE-WAY SOFTMAX TRAINING LOSS
# ==========================================================

import torch
import torch.nn.functional as F

def five_way_loss(scores, clicked_positions):
    """
    scores: [batch_size, 5]
    clicked_positions: [batch_size]

    Each group contains 1 clicked article
    and 4 non-clicked articles.
    """
    assert scores.ndim == 2
    assert scores.shape[1] == 5

    return F.cross_entropy(scores, clicked_positions)


# Test using the dummy model from Step 3
model.train()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# All five articles belong to one example impression,
# so they must share the same user history.
group_hist_cat = history_category_idx[:1].repeat(5, 1)
group_hist_subcat = history_subcategory_idx[:1].repeat(5, 1)

# Generate five scores with gradients enabled
scores = model(
    category_idx,
    subcategory_idx,
    group_hist_cat,
    group_hist_subcat,
    numerical_features
).unsqueeze(0)

# For this example, article 0 was clicked
clicked_positions = torch.tensor([0])

loss = five_way_loss(scores, clicked_positions)

# Test one optimisation step
optimizer.zero_grad()
loss.backward()
optimizer.step()

assert torch.isfinite(loss).item()

print("Scores shape:", scores.shape)
print("Training loss:", round(loss.item(), 4))
print("Five-way softmax training test PASSED!")


Scores shape: torch.Size([1, 5])
Training loss: 1.5824
Five-way softmax training test PASSED!


In [8]:

# ==========================================================
# 5. BUILD FIVE-WAY TRAINING GROUPS
# ==========================================================

import numpy as np
import pandas as pd

def build_five_way_groups(df, seed=42):
    """
    Build one-click + four-non-click training groups.

    All five articles must belong to the same impression.
    Impressions with multiple clicks generate multiple groups.
    """
    rng = np.random.default_rng(seed)
    df = df.reset_index(drop=True)

    groups = []
    targets = []
    skipped = 0

    for _, slate in df.groupby("impression_id", sort=False):

        positives = slate[slate["label"] == 1].index.to_numpy()

        negatives = (
            slate[slate["label"] == 0]
            .drop_duplicates(subset="news_id")
        )

        for positive_idx in positives:

            positive_news = df.at[positive_idx, "news_id"]

            available = negatives[
                negatives["news_id"] != positive_news
            ].index.to_numpy()

            if len(available) < 4:
                skipped += 1
                continue

            sampled_negatives = rng.choice(
                available, size=4, replace=False
            )

            five_indices = np.concatenate([
                [positive_idx], sampled_negatives
            ])

            # Shuffle so clicked article isn't always first
            order = rng.permutation(5)

            groups.append(five_indices[order])
            targets.append(int(np.where(order == 0)[0][0]))

    return (
        df,
        np.asarray(groups, dtype=np.int64).reshape(-1, 5),
        np.asarray(targets, dtype=np.int64),
        skipped
    )


# Test with dummy impression data
test_df = pd.DataFrame({
    "impression_id": [101]*7 + [102]*5 + [103]*3,
    "news_id": list("ABCDEFGHIJKLMNO"),
    "label": [
        1, 1, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0,
        1, 0, 0
    ]
})

test_rows, groups, targets, skipped = build_five_way_groups(
    test_df
)

assert groups.shape == (3, 5)
assert targets.shape == (3,)
assert skipped == 1

for indices, target in zip(groups, targets):
    selected = test_rows.iloc[indices]

    assert selected["impression_id"].nunique() == 1
    assert selected["news_id"].nunique() == 5
    assert selected["label"].sum() == 1
    assert selected["label"].iloc[target] == 1

print("Training groups:", len(groups))
print("Group shape:", groups.shape)
print("Skipped groups:", skipped)
print("Clicked positions:", targets.tolist())
print("\nFive-way grouping test PASSED!")


Training groups: 3
Group shape: (3, 5)
Skipped groups: 1
Clicked positions: [2, 1, 1]

Five-way grouping test PASSED!


In [9]:

# ==========================================================
# 6. BATCHED MLP TRAINING LOOP
# ==========================================================

import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Training device:", device)


def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0
    total_groups = 0

    for (
        cat, subcat, hist_cat, hist_subcat,
        numeric, targets
    ) in loader:

        cat = cat.to(device)
        subcat = subcat.to(device)
        hist_cat = hist_cat.to(device)
        hist_subcat = hist_subcat.to(device)
        numeric = numeric.to(device)
        targets = targets.to(device)

        batch_size = cat.shape[0]
        slate_size = cat.shape[1]

        # Each candidate in a slate shares the user's history
        repeated_hist_cat = hist_cat.repeat_interleave(
            slate_size, dim=0
        )
        repeated_hist_subcat = hist_subcat.repeat_interleave(
            slate_size, dim=0
        )

        # Score all candidates
        scores = model(
            cat.reshape(-1),
            subcat.reshape(-1),
            repeated_hist_cat,
            repeated_hist_subcat,
            numeric.reshape(-1, 5)
        ).reshape(batch_size, slate_size)

        loss = five_way_loss(scores, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item() * batch_size
        total_groups += batch_size

    return total_loss / total_groups


# ==========================================================
# TEST USING DUMMY TRAINING GROUPS
# ==========================================================

torch.manual_seed(42)

n_groups = 20
history_length = 4

dummy_cat = torch.randint(1, 20, (n_groups, 5))
dummy_subcat = torch.randint(1, 300, (n_groups, 5))

dummy_hist_cat = torch.randint(
    0, 20, (n_groups, history_length)
)
dummy_hist_subcat = torch.randint(
    0, 300, (n_groups, history_length)
)

dummy_numeric = torch.rand(n_groups, 5, 5)
dummy_targets = torch.randint(0, 5, (n_groups,))

dummy_dataset = TensorDataset(
    dummy_cat,
    dummy_subcat,
    dummy_hist_cat,
    dummy_hist_subcat,
    dummy_numeric,
    dummy_targets
)

dummy_loader = DataLoader(
    dummy_dataset,
    batch_size=4,
    shuffle=True
)

# Fresh model for the training test
test_model = FeatureMLP(
    n_categories=20,
    n_subcategories=300
).to(device)

optimizer = torch.optim.AdamW(
    test_model.parameters(),
    lr=1e-3
)

train_loss = train_one_epoch(
    test_model,
    dummy_loader,
    optimizer,
    device
)

assert np.isfinite(train_loss)

print("Training groups:", n_groups)
print("Batch size:", 4)
print("Training loss:", round(train_loss, 4))
print("\nBatched training loop PASSED!")


Training device: cuda
Training groups: 20
Batch size: 4
Training loss: 1.6244

Batched training loop PASSED!


In [10]:

# ==========================================================
# 7. TUNE AUC AND BEST MODEL CHECKPOINT
# ==========================================================

import numpy as np
import pandas as pd
import torch
from pathlib import Path


def mean_impression_auc(impression_ids, labels, scores):
    """
    Compute mean AUC across impressions.
    Impressions without both clicks and non-clicks are skipped.
    Tied scores receive average ranks.
    """

    df = pd.DataFrame({
        "impression_id": impression_ids,
        "label": labels,
        "score": scores
    })

    assert df["label"].isin([0, 1]).all()
    assert np.isfinite(df["score"]).all()

    # Rank candidate scores within each impression
    df["rank"] = df.groupby(
        "impression_id", sort=False
    )["score"].rank(method="average")

    df["positive_rank"] = df["rank"] * df["label"]

    stats = df.groupby("impression_id", sort=False).agg(
        n=("label", "size"),
        n_pos=("label", "sum"),
        pos_rank_sum=("positive_rank", "sum")
    )

    stats["n_neg"] = stats["n"] - stats["n_pos"]

    valid = stats[
        (stats["n_pos"] > 0) &
        (stats["n_neg"] > 0)
    ]

    if valid.empty:
        raise ValueError("No valid impressions for AUC.")

    aucs = (
        valid["pos_rank_sum"]
        - valid["n_pos"] * (valid["n_pos"] + 1) / 2
    ) / (valid["n_pos"] * valid["n_neg"])

    return {
        "mean_auc": float(aucs.mean()),
        "valid_impressions": len(valid),
        "skipped_impressions": len(stats) - len(valid)
    }


def save_best_checkpoint(
    model, optimizer, epoch, tune_auc, best_auc, path
):
    """Save model only when TUNE AUC improves."""

    if not np.isfinite(tune_auc):
        raise ValueError("Invalid TUNE AUC.")

    if tune_auc <= best_auc:
        return best_auc, False

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "tune_mean_auc": tune_auc,
        "architecture": "53-64-32-1"
    }, path)

    return tune_auc, True


# ==========================================================
# TEST WITH SMALL DUMMY TUNE DATA
# ==========================================================

test_impressions = ["A"] * 5 + ["B"] * 5 + ["C"] * 5

test_labels = [
    1, 0, 0, 0, 0,
    0, 1, 0, 0, 0,
    0, 0, 0, 0, 0
]

test_scores = [
    0.9, 0.1, 0.2, 0.3, 0.4,
    0.9, 0.2, 0.1, 0.4, 0.5,
    0.1, 0.2, 0.3, 0.4, 0.5
]

result = mean_impression_auc(
    test_impressions, test_labels, test_scores
)

assert np.isclose(result["mean_auc"], 0.625)
assert result["valid_impressions"] == 2
assert result["skipped_impressions"] == 1

# Test checkpoint saving in a temporary location
# This does NOT save a real trained model.
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    checkpoint_path = Path(tmp) / "test_checkpoint.pt"

    best_auc, saved = save_best_checkpoint(
        test_model,
        optimizer,
        epoch=1,
        tune_auc=result["mean_auc"],
        best_auc=-1.0,
        path=checkpoint_path
    )

    assert saved
    assert checkpoint_path.exists()

print("Mean impression AUC:", result["mean_auc"])
print("Valid impressions:", result["valid_impressions"])
print("Skipped impressions:", result["skipped_impressions"])
print("Checkpoint saving:", "PASSED")
print("\nTUNE evaluation and checkpoint test PASSED!")


Mean impression AUC: 0.625
Valid impressions: 2
Skipped impressions: 1
Checkpoint saving: PASSED

TUNE evaluation and checkpoint test PASSED!


In [11]:

# ==========================================================
# 8. PREDICT CANDIDATE RELEVANCE SCORES
# ==========================================================

import numpy as np
import pandas as pd
import torch


def predict_candidate_scores(
    model,
    category_idx,
    subcategory_idx,
    history_category_idx,
    history_subcategory_idx,
    numerical_features,
    device,
    batch_size=8192
):
    """
    Predict one relevance score per candidate article.

    All inputs must have the same number of rows.
    History arrays must be padded to equal lengths.
    """
    n = len(category_idx)

    assert len(subcategory_idx) == n
    assert len(history_category_idx) == n
    assert len(history_subcategory_idx) == n
    assert len(numerical_features) == n

    model = model.to(device)
    model.eval()

    all_scores = []

    with torch.inference_mode():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)

            cat = torch.as_tensor(
                category_idx[start:end],
                dtype=torch.long,
                device=device
            )

            subcat = torch.as_tensor(
                subcategory_idx[start:end],
                dtype=torch.long,
                device=device
            )

            hist_cat = torch.as_tensor(
                history_category_idx[start:end],
                dtype=torch.long,
                device=device
            )

            hist_subcat = torch.as_tensor(
                history_subcategory_idx[start:end],
                dtype=torch.long,
                device=device
            )

            numeric = torch.as_tensor(
                numerical_features[start:end],
                dtype=torch.float32,
                device=device
            )

            scores = model(
                cat, subcat, hist_cat, hist_subcat, numeric
            )

            all_scores.append(scores.cpu().numpy())

    return (
        np.concatenate(all_scores)
        if all_scores else np.array([], dtype=np.float32)
    )


# ==========================================================
# SMALL TEST USING EXISTING DUMMY DATA
# ==========================================================

test_scores = predict_candidate_scores(
    model=test_model,
    category_idx=dummy_cat[0].cpu().numpy(),
    subcategory_idx=dummy_subcat[0].cpu().numpy(),
    history_category_idx=np.repeat(
        dummy_hist_cat[:1].cpu().numpy(), 5, axis=0
    ),
    history_subcategory_idx=np.repeat(
        dummy_hist_subcat[:1].cpu().numpy(), 5, axis=0
    ),
    numerical_features=dummy_numeric[0].cpu().numpy(),
    device=device,
    batch_size=2
)

# Test the final score-file format
score_df = pd.DataFrame({
    "impression_id": ["dummy_1"] * 5,
    "news_id": [f"N{i}" for i in range(1, 6)],
    "score": test_scores
})

assert len(score_df) == 5
assert score_df.columns.tolist() == [
    "impression_id", "news_id", "score"
]
assert np.isfinite(score_df["score"]).all()

print(score_df)
print("\nCandidate score prediction test PASSED!")


  impression_id news_id     score
0       dummy_1      N1  0.038053
1       dummy_1      N2  0.017603
2       dummy_1      N3  0.022569
3       dummy_1      N4  0.005909
4       dummy_1      N5 -0.024181

Candidate score prediction test PASSED!


In [12]:

# ==========================================================
# 9. PREPARE FINAL MLP INPUTS
# ==========================================================

import numpy as np
import pandas as pd

# Five numerical inputs for the 53-dimensional MLP
NUMERICAL_COLUMNS = [
    "log_hist_len",
    "cat_match",
    "user_article_sim",
    "slate_exposure_pct",
    "log_article_age_hours"
]

MAX_HISTORY = 50


def prepare_mlp_inputs(
    df,
    news_category_lookup,
    news_subcategory_lookup,
    fit_mean,
    fit_std,
    max_history=MAX_HISTORY
):
    """
    Convert engineered features and click histories
    into inputs for the 53-dimensional Feature MLP.

    Numerical standardisation uses FIT statistics only.
    """

    required = [
        "category_idx",
        "subcategory_idx",
        "hist_len",
        "cat_match",
        "user_article_sim",
        "slate_exposure_pct",
        "log_article_age_hours",
        "history"
    ]

    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    n = len(df)

    # Candidate article category/subcategory indices
    category_idx = df["category_idx"].to_numpy(
        dtype=np.int64
    )

    subcategory_idx = df["subcategory_idx"].to_numpy(
        dtype=np.int64
    )

    # User click-history indices, padded to MAX_HISTORY
    hist_cat = np.zeros(
        (n, max_history), dtype=np.int64
    )

    hist_subcat = np.zeros(
        (n, max_history), dtype=np.int64
    )

    for i, history in enumerate(df["history"]):
        if not isinstance(history, str):
            continue

        # Use the most recent clicked articles
        history_ids = history.split()[-max_history:]

        for j, news_id in enumerate(history_ids):
            hist_cat[i, j] = news_category_lookup.get(
                news_id, 0
            )
            hist_subcat[i, j] = news_subcategory_lookup.get(
                news_id, 0
            )

    # Prepare the five numerical features
    numerical = np.column_stack([
        np.log1p(df["hist_len"].to_numpy(dtype=np.float32)),
        df["cat_match"].to_numpy(dtype=np.float32),
        df["user_article_sim"].to_numpy(dtype=np.float32),
        df["slate_exposure_pct"].to_numpy(dtype=np.float32),
        df["log_article_age_hours"].to_numpy(dtype=np.float32)
    ])

    # Standardise using statistics computed on FIT only
    fit_mean = np.asarray(fit_mean, dtype=np.float32).reshape(1, 5)
    fit_std = np.asarray(fit_std, dtype=np.float32).reshape(1, 5)

    if not np.all(np.isfinite(fit_mean)) or not np.all(
        np.isfinite(fit_std) & (fit_std > 0)
    ):
        raise ValueError("Invalid FIT normalisation statistics.")

    numerical = (numerical - fit_mean) / fit_std

    assert np.isfinite(numerical).all()

    return {
        "category_idx": category_idx,
        "subcategory_idx": subcategory_idx,
        "history_category_idx": hist_cat,
        "history_subcategory_idx": hist_subcat,
        "numerical_features": numerical.astype(np.float32)
    }


# ==========================================================
# SMALL INPUT-FORMAT TEST (NO REAL DATA USED)
# ==========================================================

sample_batch = pd.DataFrame({
    "category_idx": [1, 2, 3],
    "subcategory_idx": [10, 20, 30],
    "hist_len": [2, 0, 3],
    "cat_match": [0.5, 0.0, 0.33],
    "user_article_sim": [0.4, 0.0, -0.2],
    "slate_exposure_pct": [0.25, 0.5, 0.75],
    "log_article_age_hours": [1.2, 0.0, 2.0],
    "history": ["N1 N2", "", "N2 N3 N1"]
})

category_lookup = {"N1": 1, "N2": 2, "N3": 3}
subcategory_lookup = {"N1": 10, "N2": 20, "N3": 30}

# Dummy normalisation statistics only for this test
sample_inputs = prepare_mlp_inputs(
    sample_batch,
    category_lookup,
    subcategory_lookup,
    fit_mean=np.zeros(5),
    fit_std=np.ones(5)
)

assert sample_inputs["category_idx"].shape == (3,)
assert sample_inputs["subcategory_idx"].shape == (3,)
assert sample_inputs["history_category_idx"].shape == (3, 50)
assert sample_inputs["history_subcategory_idx"].shape == (3, 50)
assert sample_inputs["numerical_features"].shape == (3, 5)

# Confirm the user with no history has zero-padded inputs
assert np.all(sample_inputs["history_category_idx"][1] == 0)

print("Numerical inputs:", sample_inputs["numerical_features"].shape)
print("User history inputs:", sample_inputs["history_category_idx"].shape)
print("All input shapes correct!")
print("\nMLP input preparation PASSED!")


Numerical inputs: (3, 5)
User history inputs: (3, 50)
All input shapes correct!

MLP input preparation PASSED!


In [13]:

# ==========================================================
# 10. FIT-ONLY NUMERICAL STANDARDISATION
# ==========================================================

import numpy as np
import pyarrow.parquet as pq

RAW_NUMERIC_COLUMNS = [
    "hist_len",
    "cat_match",
    "user_article_sim",
    "slate_exposure_pct",
    "log_article_age_hours"
]


def get_numeric_matrix(df):
    """Extract the five MLP numerical features."""

    x = df[RAW_NUMERIC_COLUMNS].to_numpy(
        dtype=np.float64, copy=True
    )

    # Log-transform user history length
    x[:, 0] = np.log1p(x[:, 0])

    if not np.isfinite(x).all():
        raise ValueError("Invalid numerical feature values.")

    return x


def calculate_fit_stats(batches):
    """Calculate FIT mean/std without loading all rows."""

    count = 0
    mean = np.zeros(5, dtype=np.float64)
    m2 = np.zeros(5, dtype=np.float64)

    for df in batches:
        x = get_numeric_matrix(df)

        if len(x) == 0:
            continue

        batch_count = len(x)
        batch_mean = x.mean(axis=0)
        batch_m2 = ((x - batch_mean) ** 2).sum(axis=0)

        new_count = count + batch_count
        delta = batch_mean - mean

        mean += delta * batch_count / new_count
        m2 += batch_m2 + (
            delta ** 2 * count * batch_count / new_count
        )

        count = new_count

    if count == 0:
        raise ValueError("No FIT observations found.")

    std = np.sqrt(m2 / count)

    # Prevent division by zero for constant features
    std[std < 1e-8] = 1.0

    return mean.astype(np.float32), std.astype(np.float32)


def load_fit_stats(feature_path, batch_size=100_000):
    """Stream updated FIT features in manageable batches."""

    pf = pq.ParquetFile(feature_path)

    batches = (
        batch.to_pandas()
        for batch in pf.iter_batches(
            batch_size=batch_size,
            columns=RAW_NUMERIC_COLUMNS
        )
    )

    return calculate_fit_stats(batches)


# ==========================================================
# TEST USING DUMMY DATA ONLY
# ==========================================================

test_mean, test_std = calculate_fit_stats([sample_batch])

assert test_mean.shape == (5,)
assert test_std.shape == (5,)
assert np.isfinite(test_mean).all()
assert (test_std > 0).all()

print("FIT mean shape:", test_mean.shape)
print("FIT std shape:", test_std.shape)
print("Standardisation test PASSED!")


FIT mean shape: (5,)
FIT std shape: (5,)
Standardisation test PASSED!


In [14]:

# ==========================================================
# 11. LOAD FINAL FEATURE DATA AND INITIALISE MLP
# ==========================================================

import json
import pandas as pd
import pyarrow.parquet as pq

# Use the final validated feature files
FEATURE_PATHS = {
    split: PROCESSED_DIR / f"features_{split}.parquet"
    for split in ("fit", "tune", "eval")
}

REQUIRED_FEATURES = [
    "impression_id",
    "news_id",
    "label",
    "category_idx",
    "subcategory_idx",
    "hist_len",
    "cat_match",
    "user_article_sim",
    "slate_exposure_pct",
    "log_article_age_hours"
]

for split, path in FEATURE_PATHS.items():
    pf = pq.ParquetFile(path)

    missing = set(REQUIRED_FEATURES) - set(
        pf.schema_arrow.names
    )
    assert not missing, f"{split}: Missing {missing}"

    print(
        f"{split.upper()}: {pf.metadata.num_rows:,} rows"
    )

# Load existing category mappings from feature engineering
with open(
    PROCESSED_DIR / "step3_category_vocab.json",
    encoding="utf-8"
) as f:
    vocab = json.load(f)

category_to_idx = vocab["category_to_idx"]
subcategory_to_idx = vocab["subcategory_to_idx"]

# Load article metadata for user-history embeddings
news_metadata = pd.read_parquet(
    PROCESSED_DIR / "news_metadata.parquet",
    columns=["news_id", "category", "subcategory"]
)

news_category_lookup = {
    str(row.news_id): category_to_idx.get(
        str(row.category), 0
    )
    for row in news_metadata.itertuples(index=False)
}

news_subcategory_lookup = {
    str(row.news_id): subcategory_to_idx.get(
        str(row.subcategory), 0
    )
    for row in news_metadata.itertuples(index=False)
}

# Calculate numerical standardisation using FIT only
fit_mean, fit_std = load_fit_stats(
    FEATURE_PATHS["fit"]
)

# Initialise the actual model with the saved vocabulary
n_categories = max(category_to_idx.values()) + 1
n_subcategories = max(subcategory_to_idx.values()) + 1

model = FeatureMLP(
    n_categories=n_categories,
    n_subcategories=n_subcategories
).to(device)

print("\nFinal numerical columns:", RAW_NUMERIC_COLUMNS)
print("FIT mean:", fit_mean)
print("FIT std:", fit_std)
print("Category embedding vocabulary:", n_categories)
print("Subcategory embedding vocabulary:", n_subcategories)
print("MLP parameter count:", sum(
    p.numel() for p in model.parameters()
))
print("\nFinal Feature MLP initialised successfully!")


FIT: 12,894,210 rows
TUNE: 17,400,106 rows
EVAL: 14,085,557 rows

Final numerical columns: ['hist_len', 'cat_match', 'user_article_sim', 'slate_exposure_pct', 'log_article_age_hours']
FIT mean: [3.1359687  0.17116377 0.17823215 0.49819618 3.04584   ]
FIT std: [1.1488292  0.20033804 0.12085812 0.29212984 0.67010105]
Category embedding vocabulary: 17
Subcategory embedding vocabulary: 219
MLP parameter count: 9209

Final Feature MLP initialised successfully!


In [15]:
known_cat = sum(v != 0 for v in news_category_lookup.values())
known_subcat = sum(v != 0 for v in news_subcategory_lookup.values())

print("Articles with known categories:",
      known_cat, "/", len(news_category_lookup))
print("Articles with known subcategories:",
      known_subcat, "/", len(news_subcategory_lookup))

assert known_cat > 0
assert known_subcat > 0

print("History category lookups PASSED!")

Articles with known categories: 104148 / 104151
Articles with known subcategories: 103997 / 104151
History category lookups PASSED!


In [16]:

# ==========================================================
# 12. MEMORY-EFFICIENT IMPRESSION HISTORY LOOKUP
# ==========================================================

import sqlite3
from pathlib import Path
import pyarrow.parquet as pq

def open_history_lookup(split):
    """
    Build a temporary disk-based lookup of click histories.
    Reuse it if already completed in this Colab runtime.
    """
    if split not in ("fit", "tune", "eval"):
        raise ValueError(f"Unknown split: {split}")

    db_path = Path(f"/content/p4_history_{split}.sqlite")
    ready_path = Path(f"/content/p4_history_{split}.ready")

    if not (db_path.exists() and ready_path.exists()):
        db_path.unlink(missing_ok=True)
        ready_path.unlink(missing_ok=True)

        print(f"Indexing {split.upper()} histories...", flush=True)

        con = sqlite3.connect(str(db_path))
        con.execute("PRAGMA cache_size=-16384")
        con.execute("PRAGMA synchronous=OFF")
        con.execute("PRAGMA journal_mode=OFF")

        con.execute("""
            CREATE TABLE histories (
                impression_id TEXT PRIMARY KEY,
                history TEXT NOT NULL
            )
        """)

        pf = pq.ParquetFile(
            PROCESSED_DIR / f"{split}_impressions.parquet"
        )

        total = 0

        for batch in pf.iter_batches(
            batch_size=3000,
            columns=["impression_id", "history"]
        ):
            data = batch.to_pydict()

            rows = [
                (str(imp_id), hist if isinstance(hist, str) else "")
                for imp_id, hist in zip(
                    data["impression_id"], data["history"]
                )
            ]

            con.executemany(
                "INSERT INTO histories VALUES (?, ?)",
                rows
            )

            total += len(rows)

            if total % 100000 < len(rows):
                con.commit()
                print(f"  {total:,} histories indexed", flush=True)

        con.commit()
        con.close()
        ready_path.touch()

        print(
            f"{split.upper()} history lookup complete: {total:,}",
            flush=True
        )

    return sqlite3.connect(str(db_path))


def attach_histories(df, connection):
    """
    Attach original P1 click histories to a feature batch.
    Preserve candidate row order.
    """
    keys = df["impression_id"].astype(str)
    unique_ids = keys.unique().tolist()

    lookup = {}

    for start in range(0, len(unique_ids), 400):
        part = unique_ids[start:start + 400]
        placeholders = ",".join("?" for _ in part)

        query = (
            "SELECT impression_id, history FROM histories "
            f"WHERE impression_id IN ({placeholders})"
        )

        lookup.update(
            connection.execute(query, part).fetchall()
        )

    missing = set(unique_ids) - set(lookup)
    if missing:
        raise ValueError(
            f"Missing histories for {len(missing)} impressions"
        )

    result = df.copy()
    result["history"] = keys.map(lookup)

    return result


print("Section 12 history lookup functions defined!")


Section 12 history lookup functions defined!


In [17]:

# ==========================================================
# 13. STREAM COMPLETE FIT TRAINING GROUPS
# ==========================================================

import duckdb
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader

TRAIN_COLUMNS = [
    "impression_id",
    "news_id",
    "label",
    "category_idx",
    "subcategory_idx",
    "hist_len",
    "cat_match",
    "user_article_sim",
    "slate_exposure_pct",
    "log_article_age_hours"
]


def stream_complete_fit_batches(batch_size=10000):
    """
    Stream FIT features grouped by impression.

    Sorting ensures that one impression is never
    accidentally divided into separate training groups.
    """

    temp_dir = Path("/content/p4_mlp_temp")
    temp_dir.mkdir(exist_ok=True)

    con = duckdb.connect()
    con.execute("SET memory_limit='1GB'")
    con.execute("SET threads=2")
    con.execute(
        "SET temp_directory='/content/p4_mlp_temp'"
    )

    query = f"""
        SELECT {", ".join(TRAIN_COLUMNS)}
        FROM read_parquet(?)
        ORDER BY impression_id
    """

    pending = pd.DataFrame()

    try:
        reader = con.execute(
            query,
            [str(FEATURE_PATHS["fit"])]
        ).fetch_record_batch(rows_per_batch=batch_size)

        for arrow_batch in reader:
            df = arrow_batch.to_pandas()

            if not pending.empty:
                df = pd.concat(
                    [pending, df],
                    ignore_index=True
                )

            last_id = df["impression_id"].iloc[-1]

            # Keep the final impression for the next batch
            is_last = df["impression_id"] == last_id

            pending = df.loc[is_last].copy()
            complete = df.loc[~is_last].copy()

            if not complete.empty:
                yield complete.reset_index(drop=True)

        if not pending.empty:
            yield pending.reset_index(drop=True)

    finally:
        con.close()


def make_real_training_loader(
    df,
    history_connection,
    seed=42,
    group_batch_size=128
):
    """
    Convert complete impressions into five-way
    training groups using real saved features.
    """

    # Reuse the existing Section 5 function
    rows, groups, targets, skipped = build_five_way_groups(
        df,
        seed=seed
    )

    if len(groups) == 0:
        return None, 0, skipped

    # Reuse the Section 12 history lookup
    rows = attach_histories(rows, history_connection)

    # Reuse the Section 9 input preparation
    inputs = prepare_mlp_inputs(
        rows,
        news_category_lookup,
        news_subcategory_lookup,
        fit_mean,
        fit_std
    )

    # Five candidate articles per group
    dataset = TensorDataset(
        torch.from_numpy(
            inputs["category_idx"][groups]
        ),
        torch.from_numpy(
            inputs["subcategory_idx"][groups]
        ),
        torch.from_numpy(
            inputs["history_category_idx"][groups[:, 0]]
        ),
        torch.from_numpy(
            inputs["history_subcategory_idx"][groups[:, 0]]
        ),
        torch.from_numpy(
            inputs["numerical_features"][groups]
        ),
        torch.from_numpy(targets)
    )

    loader = DataLoader(
        dataset,
        batch_size=group_batch_size,
        shuffle=True,
        num_workers=0
    )

    return loader, len(groups), skipped


print("Section 13 real FIT training functions ready!")


Section 13 real FIT training functions ready!


In [18]:

# ==========================================================
# 14. MEMORY-EFFICIENT TUNE VALIDATION
# ==========================================================

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

TUNE_SUBSET_PATH = Path("/content/p4_fixed_tune.parquet")
TUNE_SAMPLE_SIZE = 5000


def prepare_fixed_tune_subset():
    """
    Select complete TUNE impressions once.
    Save locally so every epoch uses the same examples.
    """

    if TUNE_SUBSET_PATH.exists():
        print("Reusing existing fixed TUNE subset.")
        return

    print("Selecting fixed TUNE impressions...", flush=True)

    pf_impressions = pq.ParquetFile(
        PROCESSED_DIR / "tune_impressions.parquet"
    )

    impression_ids = []

    for batch in pf_impressions.iter_batches(
        batch_size=50000,
        columns=["impression_id"]
    ):
        impression_ids.extend(
            str(x) for x in batch.column(0).to_pylist()
        )

    rng = np.random.default_rng(42)

    selected = set(
        rng.choice(
            np.array(impression_ids, dtype=object),
            size=min(TUNE_SAMPLE_SIZE, len(impression_ids)),
            replace=False
        ).tolist()
    )

    del impression_ids

    print(
        f"Selected {len(selected):,} impressions.",
        flush=True
    )

    pf = pq.ParquetFile(FEATURE_PATHS["tune"])

    schema = pa.schema([
        pf.schema_arrow.field(col)
        for col in TRAIN_COLUMNS
    ])

    temp_path = TUNE_SUBSET_PATH.with_suffix(".partial.parquet")
    temp_path.unlink(missing_ok=True)

    rows_saved = 0
    found_ids = set()

    writer = pq.ParquetWriter(
        temp_path,
        schema,
        compression="snappy"
    )

    try:
        for batch in pf.iter_batches(
            batch_size=50000,
            columns=TRAIN_COLUMNS
        ):
            df = batch.to_pandas()

            mask = df["impression_id"].astype(str).isin(
                selected
            )

            if not mask.any():
                continue

            part = df.loc[mask, TRAIN_COLUMNS].copy()

            found_ids.update(
                part["impression_id"].astype(str).unique()
            )

            writer.write_table(
                pa.Table.from_pandas(
                    part,
                    schema=schema,
                    preserve_index=False
                )
            )

            rows_saved += len(part)

    finally:
        writer.close()

    if found_ids != selected:
        raise ValueError(
            "Some selected TUNE impressions were missing."
        )

    temp_path.replace(TUNE_SUBSET_PATH)

    print(
        f"Fixed TUNE subset saved: {rows_saved:,} rows",
        flush=True
    )


def evaluate_fixed_tune(model):
    """
    Evaluate mean impression AUC on fixed complete
    TUNE impressions without loading all TUNE data.
    """

    prepare_fixed_tune_subset()

    history_con = open_history_lookup("tune")

    pf = pq.ParquetFile(TUNE_SUBSET_PATH)

    all_ids = []
    all_labels = []
    all_scores = []

    try:
        for batch in pf.iter_batches(
            batch_size=5000,
            columns=TRAIN_COLUMNS
        ):
            df = batch.to_pandas()

            df = attach_histories(
                df, history_con
            )

            inputs = prepare_mlp_inputs(
                df,
                news_category_lookup,
                news_subcategory_lookup,
                fit_mean,
                fit_std
            )

            scores = predict_candidate_scores(
                model=model,
                category_idx=inputs["category_idx"],
                subcategory_idx=inputs["subcategory_idx"],
                history_category_idx=inputs[
                    "history_category_idx"
                ],
                history_subcategory_idx=inputs[
                    "history_subcategory_idx"
                ],
                numerical_features=inputs[
                    "numerical_features"
                ],
                device=device,
                batch_size=2048
            )

            all_ids.extend(
                df["impression_id"].tolist()
            )
            all_labels.extend(
                df["label"].tolist()
            )
            all_scores.append(scores)

    finally:
        history_con.close()

    result = mean_impression_auc(
        all_ids,
        all_labels,
        np.concatenate(all_scores)
    )

    return result


print("Section 14 TUNE validation functions ready!")


Section 14 TUNE validation functions ready!


In [6]:

# ==========================================================
# 15. REAL FIT TRAINING WITH TUNE MODEL SELECTION
# ==========================================================

import os
import time
import gc
import duckdb
import numpy as np
import torch
from pathlib import Path

MAX_EPOCHS = 5
PATIENCE = 2
LEARNING_RATE = 1e-3

CHECKPOINT_DIR = PROCESSED_DIR.parent / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "step3_mlp_best_53.pt"
LAST_MODEL_PATH = CHECKPOINT_DIR / "step3_mlp_last_53.pt"


def train_real_feature_mlp():
    """
    Train using streamed FIT impressions and five-way groups.

    - FIT: model training
    - TUNE: checkpoint selection
    - Saves completed epochs for restart recovery
    """

    optimizer_real = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    best_auc = -np.inf
    bad_epochs = 0
    start_epoch = 1

    # Resume only from a compatible completed epoch.
    if LAST_MODEL_PATH.exists():
        state = torch.load(
            LAST_MODEL_PATH,
            map_location=device,
            weights_only=False
        )

        if (
            state["architecture"] != "53-64-32-1"
            or state["feature_names"] != RAW_NUMERIC_COLUMNS
            or state["vocab_sizes"] != [
                n_categories, n_subcategories
            ]
        ):
            raise ValueError(
                "Saved checkpoint does not match the current model."
            )

        model.load_state_dict(state["model_state_dict"])
        optimizer_real.load_state_dict(
            state["optimizer_state_dict"]
        )

        start_epoch = state["epoch"] + 1
        best_auc = state["best_auc"]
        bad_epochs = state["bad_epochs"]

        print(
            f"Resuming after completed epoch {state['epoch']}"
        )

    elif BEST_MODEL_PATH.exists():
        raise RuntimeError(
            "A best checkpoint exists but no last-epoch "
            "checkpoint was found. Check the existing files "
            "before starting a new training run."
        )

    if start_epoch > MAX_EPOCHS or bad_epochs >= PATIENCE:
        print("Training is already complete.")
        print("Best TUNE AUC:", best_auc)
        print("Best checkpoint:", BEST_MODEL_PATH)
        return

    # Build the FIT history lookup once.
    history_connection = open_history_lookup("fit")

    try:
        for epoch in range(start_epoch, MAX_EPOCHS + 1):
            print(
                f"\n========== EPOCH {epoch}/{MAX_EPOCHS} ==========",
                flush=True
            )

            started = time.time()

            total_loss = 0.0
            total_groups = 0
            total_skipped = 0
            total_rows = 0

            for batch_number, df in enumerate(
                stream_complete_fit_batches(
                    batch_size=10000
                ),
                start=1
            ):
                total_rows += len(df)

                loader, n_groups, skipped = (
                    make_real_training_loader(
                        df,
                        history_connection,
                        seed=42 + epoch * 100000 + batch_number,
                        group_batch_size=128
                    )
                )

                total_skipped += skipped

                if loader is not None:
                    batch_loss = train_one_epoch(
                        model,
                        loader,
                        optimizer_real,
                        device
                    )

                    total_loss += batch_loss * n_groups
                    total_groups += n_groups

                del loader, df

                if batch_number % 50 == 0:
                    print(
                        f"FIT rows: {total_rows:,} | "
                        f"Training groups: {total_groups:,} | "
                        f"Skipped: {total_skipped:,}",
                        flush=True
                    )

            expected_rows = pq.ParquetFile(
                FEATURE_PATHS["fit"]
            ).metadata.num_rows

            if total_rows != expected_rows:
                raise ValueError(
                    f"FIT incomplete: {total_rows:,} "
                    f"of {expected_rows:,} rows."
                )

            if total_groups == 0:
                raise ValueError(
                    "No valid five-way training groups generated."
                )

            epoch_loss = total_loss / total_groups

            print(
                f"\nFIT epoch complete. "
                f"Average loss: {epoch_loss:.4f}",
                flush=True
            )

            # Evaluate on the same fixed TUNE subset.
            tune_results = evaluate_fixed_tune(model)
            tune_auc = float(tune_results["mean_auc"])

            print(
                f"TUNE mean AUC: {tune_auc:.4f}",
                flush=True
            )
            print(
                "Valid TUNE impressions:",
                tune_results["valid_impressions"]
            )

            if not np.isfinite(tune_auc):
                raise ValueError("Invalid TUNE AUC.")

            # Save the best-performing model.
            if tune_auc > best_auc + 1e-4:
                best_auc, _ = save_best_checkpoint(
                    model=model,
                    optimizer=optimizer_real,
                    epoch=epoch,
                    tune_auc=tune_auc,
                    best_auc=best_auc,
                    path=BEST_MODEL_PATH
                )

                bad_epochs = 0
                print("New best model saved!", flush=True)
            else:
                bad_epochs += 1
                print(
                    f"No improvement: {bad_epochs}/{PATIENCE}",
                    flush=True
                )

            # Save the last completed epoch for resuming.
            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer_real.state_dict(),
                "best_auc": best_auc,
                "bad_epochs": bad_epochs,
                "architecture": "53-64-32-1",
                "feature_names": list(RAW_NUMERIC_COLUMNS),
                "vocab_sizes": [
                    n_categories, n_subcategories
                ]
            }

            temp_checkpoint = CHECKPOINT_DIR / (
                "step3_mlp_last_53.partial.pt"
            )

            torch.save(checkpoint, temp_checkpoint)
            os.replace(temp_checkpoint, LAST_MODEL_PATH)

            elapsed_minutes = (time.time() - started) / 60

            print(
                f"Epoch {epoch} complete | "
                f"Loss: {epoch_loss:.4f} | "
                f"TUNE AUC: {tune_auc:.4f} | "
                f"Elapsed: {elapsed_minutes:.1f} min",
                flush=True
            )

            gc.collect()

            if bad_epochs >= PATIENCE:
                print("Early stopping triggered.")
                break

    finally:
        history_connection.close()

    print("\nTRAINING COMPLETE!")
    print("Best TUNE mean AUC:", round(best_auc, 4))
    print("Best checkpoint:", BEST_MODEL_PATH)


print("Section 15 real FIT training function ready!")


AttributeError: partially initialized module 'torch' from '/usr/local/lib/python3.13/dist-packages/torch/__init__.py' has no attribute 'fx' (most likely due to a circular import)

In [1]:
# RUN REAL FEATURE MLP TRAINING
train_real_feature_mlp()

NameError: name 'train_real_feature_mlp' is not defined

In [2]:

# ==========================================================
# 16. SCORE FULL EVAL WITH THE BEST TUNE CHECKPOINT
# ==========================================================

import gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import pyarrow as pa
import pyarrow.parquet as pq

EVAL_PRED_DIR = (
    PROCESSED_DIR.parent / "results" / "step3_feature_mlp"
)
EVAL_PRED_DIR.mkdir(parents=True, exist_ok=True)

EVAL_PRED_PATH = EVAL_PRED_DIR / "eval_predictions_53.parquet"

EVAL_INPUT_COLUMNS = TRAIN_COLUMNS + ["cold_article"]


def generate_final_eval_predictions(batch_size=10000):
    # Load the BEST checkpoint, not the last epoch.
    if not BEST_MODEL_PATH.exists():
        raise FileNotFoundError(
            "Best TUNE checkpoint missing. Finish Section 15 first."
        )

    state = torch.load(
        BEST_MODEL_PATH,
        map_location=device,
        weights_only=False
    )

    if state.get("architecture") != "53-64-32-1":
        raise ValueError("Checkpoint is not the new 53-input MLP.")

    final_model = FeatureMLP(
        n_categories,
        n_subcategories
    ).to(device)

    final_model.load_state_dict(
        state["model_state_dict"],
        strict=True
    )
    final_model.eval()

    source = pq.ParquetFile(FEATURE_PATHS["eval"])
    expected_rows = source.metadata.num_rows

    temp_path = EVAL_PRED_DIR / (
        "eval_predictions_53.partial.parquet"
    )
    temp_path.unlink(missing_ok=True)

    # Read original clicked histories using Section 12.
    history_connection = open_history_lookup("eval")

    writer = None
    rows_written = 0

    print(
        f"Starting full EVAL prediction: {expected_rows:,} rows",
        flush=True
    )

    try:
        for batch in source.iter_batches(
            batch_size=batch_size,
            columns=EVAL_INPUT_COLUMNS
        ):
            frame = attach_histories(
                batch.to_pandas(),
                history_connection
            )

            inputs = prepare_mlp_inputs(
                frame,
                news_category_lookup,
                news_subcategory_lookup,
                fit_mean,
                fit_std
            )

            scores = predict_candidate_scores(
                final_model,
                inputs["category_idx"],
                inputs["subcategory_idx"],
                inputs["history_category_idx"],
                inputs["history_subcategory_idx"],
                inputs["numerical_features"],
                device,
                batch_size=2048
            )

            if (
                len(scores) != len(frame)
                or not np.isfinite(scores).all()
            ):
                raise ValueError("Invalid EVAL scores.")

            output = frame[
                ["impression_id", "news_id", "label", "cold_article"]
            ].copy()

            output["score"] = scores.astype(np.float32)

            table = pa.Table.from_pandas(
                output,
                preserve_index=False
            )

            if writer is None:
                writer = pq.ParquetWriter(
                    temp_path,
                    table.schema,
                    compression="zstd"
                )

            writer.write_table(table)
            rows_written += len(output)

            if rows_written % 500000 < len(output):
                print(
                    f"EVAL scored: {rows_written:,}/{expected_rows:,}",
                    flush=True
                )

            del frame, inputs, scores, output, table

    finally:
        history_connection.close()

        if writer is not None:
            writer.close()

        gc.collect()

    # Validate before replacing any previous prediction file.
    if (
        rows_written != expected_rows
        or pq.ParquetFile(temp_path).metadata.num_rows
        != expected_rows
    ):
        raise ValueError(
            f"EVAL incomplete: {rows_written:,}/{expected_rows:,}"
        )

    temp_path.replace(EVAL_PRED_PATH)

    print("\nFINAL EVAL PREDICTIONS SAVED!")
    print("Rows:", f"{rows_written:,}")
    print("Selected checkpoint epoch:", state["epoch"])
    print("Output:", EVAL_PRED_PATH)


print("Section 16 ready!")


KeyboardInterrupt: 

In [ ]:

# ==========================================================
# 17. FINAL EVAL METRICS AND P2 SCORE EXPORT
# ==========================================================

from utils import metrics as metrics_module
import duckdb

P2_SCORE_PATH = (
    PROCESSED_DIR.parent / "scores"
    / "step3_feature_mlp.parquet"
)

METRIC_NAMES = ("auc", "mrr", "ndcg@5", "ndcg@10")


def evaluate_and_export_final_scores():
    if not EVAL_PRED_PATH.exists():
        raise FileNotFoundError(
            "Run Section 16 EVAL predictions first."
        )

    source = pq.ParquetFile(EVAL_PRED_PATH)

    expected_rows = pq.ParquetFile(
        FEATURE_PATHS["eval"]
    ).metadata.num_rows

    if source.metadata.num_rows != expected_rows:
        raise ValueError("Incorrect EVAL prediction row count.")

    metric_sums = {
        name: 0.0 for name in METRIC_NAMES
    }
    metric_counts = {
        name: 0 for name in METRIC_NAMES
    }

    evaluated_impressions = 0
    closed_impressions = set()
    pending = None
    seen_rows = 0

    def process_complete(frame):
        nonlocal evaluated_impressions

        if frame.empty:
            return

        ids = set(
            frame["impression_id"].astype(str).unique()
        )

        if ids & closed_impressions:
            raise ValueError(
                "Impressions are not contiguous in the "
                "prediction file. Evaluation must be regrouped."
            )

        closed_impressions.update(ids)

        results = metrics_module.evaluate(
            frame,
            score_col="score",
            label_col="label",
            group_col="impression_id",
            per_impression=True
        )

        evaluated_impressions += len(results)

        for name in METRIC_NAMES:
            values = pd.to_numeric(
                results[name],
                errors="coerce"
            ).to_numpy(dtype=np.float64)

            valid = np.isfinite(values)

            metric_sums[name] += float(
                values[valid].sum()
            )
            metric_counts[name] += int(valid.sum())

    # Calculate metrics using complete impressions.
    for batch_no, batch in enumerate(
        source.iter_batches(
            batch_size=50000,
            columns=["impression_id", "label", "score"]
        ),
        start=1
    ):
        df = batch.to_pandas()
        seen_rows += len(df)

        combined = (
            pd.concat([pending, df], ignore_index=True)
            if pending is not None else df
        )

        last_id = combined["impression_id"].iloc[-1]

        last_start = int(
            np.flatnonzero(
                combined["impression_id"].to_numpy()
                == last_id
            )[0]
        )

        process_complete(combined.iloc[:last_start])

        pending = combined.iloc[last_start:].copy()

        if batch_no % 25 == 0:
            print(
                f"EVAL metrics: {seen_rows:,}/{expected_rows:,} rows",
                flush=True
            )

    if pending is not None:
        process_complete(pending)

    summary = {
        name: (
            metric_sums[name] / metric_counts[name]
            if metric_counts[name] else float("nan")
        )
        for name in METRIC_NAMES
    }

    summary["n_impressions"] = len(closed_impressions)
    summary["n_skipped"] = (
        len(closed_impressions) - evaluated_impressions
    )
    summary["n_candidates"] = seen_rows

    if (
        seen_rows != expected_rows
        or not all(
            np.isfinite(summary[name])
            for name in METRIC_NAMES
        )
    ):
        raise ValueError("EVAL metric validation failed.")

    print("\nFINAL EVAL METRICS")
    print(pd.Series(summary))

    # Save a small summary for the final report.
    pd.DataFrame([summary]).to_csv(
        EVAL_PRED_DIR / "eval_summary_53.csv",
        index=False
    )

    # ----------------------------------------------
    # EXPORT SCORES IN ORIGINAL P1 CANDIDATE ORDER
    # ----------------------------------------------

    P2_SCORE_PATH.parent.mkdir(
        parents=True, exist_ok=True
    )

    temp_path = P2_SCORE_PATH.with_name(
        "step3_feature_mlp.partial.parquet"
    )
    temp_path.unlink(missing_ok=True)

    temp_dir = Path("/content/p4_eval_sort")
    temp_dir.mkdir(exist_ok=True)

    con = duckdb.connect()
    con.execute("SET memory_limit='1GB'")
    con.execute("SET threads=2")
    con.execute(
        "SET temp_directory='/content/p4_eval_sort'"
    )

    # Use the original P1 candidate order, rather than
    # assuming P4's feature-file rows have identical order.
    original_candidates = (
        PROCESSED_DIR / "eval_candidates.parquet"
    )

    query = """
        WITH original AS (
            SELECT
                ROW_NUMBER() OVER () AS original_row,
                impression_id,
                news_id
            FROM read_parquet(?)
        )
        SELECT
            original.impression_id,
            original.news_id,
            scored.score
        FROM original
        LEFT JOIN read_parquet(?) AS scored
          ON CAST(original.impression_id AS VARCHAR)
             = CAST(scored.impression_id AS VARCHAR)
         AND CAST(original.news_id AS VARCHAR)
             = CAST(scored.news_id AS VARCHAR)
        ORDER BY original.original_row
    """

    writer = None
    exported = 0

    try:
        reader = con.execute(
            query,
            [
                str(original_candidates),
                str(EVAL_PRED_PATH)
            ]
        ).fetch_record_batch(rows_per_batch=50000)

        for batch in reader:
            score_values = batch.column(2).to_numpy(
                zero_copy_only=False
            )

            if not np.isfinite(score_values).all():
                raise ValueError(
                    "Missing or invalid score during export."
                )

            if writer is None:
                writer = pq.ParquetWriter(
                    temp_path,
                    batch.schema,
                    compression="zstd"
                )

            writer.write_batch(batch)
            exported += len(batch)

    finally:
        if writer is not None:
            writer.close()
        con.close()

    # Validate file before publishing it for P2.
    check = pq.ParquetFile(temp_path)

    if (
        exported != expected_rows
        or check.metadata.num_rows != expected_rows
    ):
        raise ValueError(
            "Export row-count mismatch or duplicate keys."
        )

    if check.schema_arrow.names != [
        "impression_id", "news_id", "score"
    ]:
        raise ValueError("Incorrect P2 score-file columns.")

    # Publish only after successful validation.
    temp_path.replace(P2_SCORE_PATH)

    print("\nFINAL P2 SCORE FILE READY!")
    print("Candidates:", f"{exported:,}")
    print("Output:", P2_SCORE_PATH)


print("Section 17 ready!")


In [ ]:
generate_final_eval_predictions()

In [ ]:
evaluate_and_export_final_scores()